In [1]:
# Without the import:
def fix_engine(e: Engine):  # Python says: "Stop! I need to know what 'Engine' is RIGHT NOW to define this."
    pass

# Result: NameError: name 'Engine' is not defined.

NameError: name 'Engine' is not defined

In [2]:
from __future__ import annotations

def fix_engine(e: Engine):  # Python says: "Okay, I'll just store 'Engine' as a string. Moving on!"
    pass

class Engine:
    pass

# Result: Success!

In [4]:
def set_status(status: str):
    if status == "active":
        print("System is up!")

set_status("active") # OOPS! Typo. The code runs but nothing happens.

System is up!


In [8]:
from enum import Enum

class Status(Enum):
    ACTVE = "active"
    INACTIVE = "inactive"
    MAINTENANCE = "maintenance"

def set_status(status: Status):
    if status == Status.ACTVE:
        print("System is up!")

# This is safe. You can't typo 'Status.ACTIVE' without the computer 
# immediately telling you "Hey, that doesn't exist!"
set_status(Status.ACTVE)

System is up!


In [2]:
import time
import asyncio
from typing import Any

# --- THE STRATEGY WE DISCUSSED ---
class CacheStrategy:
    def __init__(self, max_age_seconds: float = 5.0): # Short 5-second window
        self._max_age_seconds = max_age_seconds
        self._cache = {}

    def store(self, breaker_name: str, state: dict[str, Any]):
        # The "Photocopy and Timestamp" step
        self._cache[breaker_name] = (dict(state), time.monotonic())
        print(f"--- [STORED] Backup updated for {breaker_name} ---")

    async def execute(self, state, breaker_name):
        # The "Emergency Search" step
        if breaker_name not in self._cache:
            print(f"--- [MISS] No backup found for {breaker_name} ---")
            fallback = dict(state)
            fallback["__cascadebreaker_cache_miss__"] = True
            return fallback

        # If found, check the age
        cached_data, timestamp = self._cache[breaker_name]
        age = time.monotonic() - timestamp
        
        # Add the "From Cache" label
        result = dict(cached_data)
        result["__from_cache__"] = True
        result["age"] = f"{age:.1f}s"
        
        print(f"--- [HIT] Found backup! Age: {age:.1f}s ---")
        return result

# --- USING IT IN A REAL SCENARIO ---
async def main():
    strategy = CacheStrategy(max_age_seconds=5.0)
    service_name = "ProductService"

    # 1. SUCCESS: The app is working perfectly
    print("\nStep 1: System is healthy...")
    live_data = {"products": ["Laptop", "Mouse"], "price": 1200}
    strategy.store(service_name, live_data)

    # 2. WAIT: Time passes...
    print("Waiting 2 seconds...")
    await asyncio.sleep(2)

    # 3. FAILURE: The internet dies!
    print("\nStep 2: Internet fails! Calling execute()...")
    backup_data = await strategy.execute({}, service_name)
    print(f"Data shown to user: {backup_data}")

    # 4. MISS SCENARIO: Try a service we NEVER stored
    print("\nStep 3: Checking a service with NO backup...")
    empty_result = await strategy.execute({}, "UnknownService")
    print(f"Data shown to user: {empty_result}")

if __name__ == "__main__":
    await main()


Step 1: System is healthy...
--- [STORED] Backup updated for ProductService ---
Waiting 2 seconds...

Step 2: Internet fails! Calling execute()...
--- [HIT] Found backup! Age: 2.0s ---
Data shown to user: {'products': ['Laptop', 'Mouse'], 'price': 1200, '__from_cache__': True, 'age': '2.0s'}

Step 3: Checking a service with NO backup...
--- [MISS] No backup found for UnknownService ---
Data shown to user: {'__cascadebreaker_cache_miss__': True}


In [3]:
import time
import asyncio
from enum import Enum
from typing import Any, NamedTuple
from dataclasses import dataclass

# --- 1. MOCKING THE BASE INFRASTRUCTURE ---
class FallbackStrategy(Enum):
    CACHE = "cache"

@dataclass
class FallbackResult:
    state: dict[str, Any]
    strategy_used: FallbackStrategy
    confidence: float
    latency_ms: float
    metadata: dict[str, Any]

class BaseFallbackStrategy:
    pass

# --- 2. YOUR CLASS (WITH THE LOGIC YOU PROVIDED) ---
class CacheStrategy(BaseFallbackStrategy):
    def __init__(self, max_age_seconds: float = 300.0) -> None:
        self._max_age_seconds = max_age_seconds
        self._cache: dict[str, tuple[dict[str, Any], float]] = {}

    def store(self, breaker_name: str, state: dict[str, Any]) -> None:
        # Saving the snapshot and the timestamp
        self._cache[breaker_name] = (dict(state), time.monotonic())

    async def execute(
        self,
        state: dict[str, Any],
        breaker_name: str,
        failure_context: dict[str, Any],
    ) -> FallbackResult:
        t0 = time.monotonic()

        # CHECK 1: Is the locker empty? (Cache Miss)
        if breaker_name not in self._cache:
            fallback_state = dict(state)
            fallback_state["__cascadebreaker_cache_miss__"] = True
            return FallbackResult(
                state=fallback_state,
                strategy_used=FallbackStrategy.CACHE,
                confidence=0.0,
                latency_ms=(time.monotonic() - t0) * 1000,
                metadata={"cache_hit": False},
            )

        # CHECK 2: Found it! Let's check the age.
        cached_state, cached_at = self._cache[breaker_name]
        age_seconds = time.monotonic() - cached_at
        
        # If it's older than our limit, it's "Stale"
        is_stale = age_seconds > self._max_age_seconds
        
        # Confidence logic: 0.8 if fresh, 0.2 if old/stale
        confidence = 0.2 if is_stale else 0.8

        returned_state = dict(cached_state)
        returned_state["__cascadebreaker_from_cache__"] = True

        return FallbackResult(
            state=returned_state,
            strategy_used=FallbackStrategy.CACHE,
            confidence=confidence,
            latency_ms=(time.monotonic() - t0) * 1000,
            metadata={"cache_hit": True, "age_seconds": round(age_seconds, 1)},
        )

# --- 3. RUNNING THE EXAMPLE ---
async def main():
    # Set a very short max age of 2 seconds for this demo
    strategy = CacheStrategy(max_age_seconds=2.0)
    service = "WeatherAPI"

    print("--- Step 1: Normal Operation ---")
    # We get good data and store it
    live_data = {"temp": 72, "city": "New York"}
    strategy.store(service, live_data)
    print("Stored New York weather data.")

    print("\n--- Step 2: Immediate Failure (Fresh Data) ---")
    # The API breaks immediately
    result = await strategy.execute({}, service, {"error": "Timeout"})
    print(f"Confidence: {result.confidence} (Expect 0.8)")
    print(f"Data: {result.state}")

    print("\n--- Step 3: Waiting for data to get stale... ---")
    await asyncio.sleep(3) # Wait longer than our 2s max_age

    print("\n--- Step 4: Delayed Failure (Stale Data) ---")
    stale_result = await strategy.execute({}, service, {"error": "Timeout"})
    print(f"Confidence: {stale_result.confidence} (Expect 0.2 because it's > 2s old)")
    print(f"Metadata: {stale_result.metadata}")

# If in a notebook, use: await main()
# If in a .py file, use: asyncio.run(main())
await main()

--- Step 1: Normal Operation ---
Stored New York weather data.

--- Step 2: Immediate Failure (Fresh Data) ---
Confidence: 0.8 (Expect 0.8)
Data: {'temp': 72, 'city': 'New York', '__cascadebreaker_from_cache__': True}

--- Step 3: Waiting for data to get stale... ---

--- Step 4: Delayed Failure (Stale Data) ---
Confidence: 0.2 (Expect 0.2 because it's > 2s old)
Metadata: {'cache_hit': True, 'age_seconds': 3.0}


In [9]:
def flatten_list(nested_list):
    flat_list=[]
    for i in nested_list:
        if isinstance(i,list):
            print(i)
            flat_list.extend(flatten_list(i))
            print(flat_list)
        else:
            flat_list.append(i)

    
    return flat_list

In [11]:
ip= [1,2,[3]]

flatten_list(ip)

[3]
[1, 2, 3]


[1, 2, 3]